# Encrypted Math Tutorial: Special Functions (`math/special.py`)

This tutorial covers `src/concrete_fhe_toolkit/math/special.py`. It provides fixed-point approximations for complex transcendental functions like Trigonometry (sin, cos), Exponentials (exp, log), and special math functions (sqrt, erf) over encrypted data.

## 1. Trigonometry (`make_sin`, `make_cos`)
We evaluate trigonometric functions over a scaled domain. Here we use an input scale of 100 and output scale of 1000.

In [ ]:
from concrete import fhe
from concrete_fhe_toolkit.math.special import make_sin, make_cos

# Input scale 100 (e.g. 314 = 3.14 radians)
sin_fn = make_sin(min_input=0, max_input=628, input_scale=100, output_scale=1000)
cos_fn = make_cos(min_input=0, max_input=628, input_scale=100, output_scale=1000)

def test_trig(angle: int):
    return sin_fn(angle), cos_fn(angle)

assert test_trig(0) == (0, 1000) # sin(0)=0, cos(0)=1 (scaled by 1000)
assert test_trig(314) == (2, -1000) # sin(pi) approx 0, cos(pi) approx -1
print("Cleartext trig passed!")

compiler = fhe.Compiler(test_trig, {"angle": "encrypted"})
inputset = [(0,), (314,), (157,)]
circuit = compiler.compile(inputset)

enc_res = circuit.encrypt_run_decrypt(0)
assert enc_res == (0, 1000)
print("✅ Encrypted trig passed!")

## 2. Exponentials and Logarithms (`make_exp`, `make_log`)

In [ ]:
from concrete_fhe_toolkit.math.special import make_exp, make_log

exp_fn = make_exp(min_input=-50, max_input=50, input_scale=10, output_scale=100)
log_fn = make_log(min_input=1, max_input=1000, input_scale=100, output_scale=100)

def test_explog(e_val: int, l_val: int):
    return exp_fn(e_val), log_fn(l_val)

assert test_explog(0, 100) == (100, 0) # exp(0)=1 (scaled 100), log(1)=0
print("Cleartext explog passed!")

compiler = fhe.Compiler(test_explog, {"e_val": "encrypted", "l_val": "encrypted"})
inputset = [(0, 100), (-10, 500)]
circuit = compiler.compile(inputset)

enc_res = circuit.encrypt_run_decrypt(0, 100)
assert enc_res == (100, 0)
print("✅ Encrypted explog passed!")

## 3. Roots (`make_sqrt`)

In [ ]:
from concrete_fhe_toolkit.math.special import make_sqrt

sqrt_fn = make_sqrt(min_input=0, max_input=1000, input_scale=100, output_scale=100)

def test_sqrt(val: int):
    return sqrt_fn(val)

# sqrt(4.00) = 2.00 -> 400 yields 200
assert test_sqrt(400) == 200
print("Cleartext sqrt passed!")

compiler = fhe.Compiler(test_sqrt, {"val": "encrypted"})
inputset = [(0,), (100,), (400,)]
circuit = compiler.compile(inputset)

enc_res = circuit.encrypt_run_decrypt(400)
assert enc_res == 200
print("✅ Encrypted sqrt passed!")